# Session 4 · Errors and debugging

Work top to bottom and run each cell with **Shift+Enter**.

Cells under a **Your turn** heading contain `...` placeholders. Replace them, run the cell, then run the check cell beneath it. Checks print `PASS` or `NOT YET` and never stop the notebook.

Solutions are in the matching `_solutions` notebook. Open it only after a genuine attempt.

**This notebook is different:** many cells are *meant* to fail. Run it **one cell at a time**, not with Run All, because Run All stops at the first error. Read each error, then fix the code in that cell.

In [1]:
def check(label, get, expected):
    """Print PASS or NOT YET. `get` is a lambda, so missing names or errors never stop the notebook."""
    try:
        got = get()
    except Exception as e:
        print(f"NOT YET  {label}: {type(e).__name__}: {e}")
        return
    try:
        ok = got.equals(expected) if hasattr(got, "equals") else bool(got == expected)
        if isinstance(expected, float) and isinstance(got, (int, float)):
            ok = abs(got - expected) < 1e-6
    except Exception:
        ok = False
    print(f"PASS     {label}" if ok else f"NOT YET  {label}: expected {expected!r}, got {got!r}")

## 1. Reading a traceback

Read a traceback from the **bottom up**:

1. The last line names the error type and gives a message. Start there.
2. The lines above show the chain of calls that led to it, like R's `traceback()`, with the failing line marked by an arrow.
3. The bug is usually in *your* code nearest the bottom, not in library code.

Run the next cell. It fails on purpose.

In [2]:
def quarter_average(scores):
    return sum(scores) / len(scores)


def team_summary(team, scores):
    avg = quarter_average(scores)
    return f"{team} averaged {avg:.1f} goals a quarter"


team_summary("Geelong", [])

ZeroDivisionError: division by zero

### Your turn 1.1

The bottom line says `ZeroDivisionError`, and the arrow points into `quarter_average()`: an empty list has length 0. Fix `quarter_average()` so it returns `0.0` for an empty list and still works normally otherwise.

In [5]:
def quarter_average(scores):
    return sum(scores) / len(scores) if scores else 0.0 

In [6]:
check("1.1 empty list", lambda: quarter_average([]), 0.0)
check("1.1 normal list", lambda: quarter_average([3, 5, 4, 3]), 3.75)

PASS     1.1 empty list
PASS     1.1 normal list


## 2. The error gallery

Each cell below fails with a different, very common error. Run it, read the last line of the error, fix the cell, and run it again. Then run its check.

### 2.1 NameError: a name Python has never seen
R says `object 'scroe' not found`.

In [8]:
score = 102
message = f"Final score: {score}"

In [9]:
check("2.1", lambda: message, "Final score: 102")

PASS     2.1


### 2.2 TypeError: the wrong type for the operation
Python will not silently turn a number into text. R's `paste()` does that conversion for you; `+` does not.

In [11]:
label = "Score: " + "102"

In [12]:
check("2.2", lambda: label, "Score: 102")

PASS     2.2


### 2.3 KeyError: a dictionary key that does not exist
R returns `NULL` for a missing list element. Python raises an error. Use `.get()` with a default of 0.

In [16]:
ladder = {"Geelong": 60, "Sydney": 64, "Carlton": 0}
carlton = ladder["Carlton"]

In [17]:
check("2.3", lambda: carlton, 0)

PASS     2.3


### 2.4 IndexError: a position past the end
R returns `NA` for `x[5]` on a length-4 vector. Python raises an error. Set `last_quarter` to the last element.

In [20]:
quarters = [3, 5, 4, 3]
last_quarter = quarters[3]

In [21]:
check("2.4", lambda: last_quarter, 3)

PASS     2.4


### 2.5 AttributeError: a method the object does not have
The message tells you the type (`list`) and the missing name. Lists have `.append()`, not `.push()`.

In [23]:
teams = ["Geelong"]
teams.append("Sydney")

In [24]:
check("2.5", lambda: teams, ["Geelong", "Sydney"])

PASS     2.5


### 2.6 ValueError: right type, unusable value
`int()` accepts text, but not text that looks like `15.12`. Extract the goals (the part before the dot) as an integer.

In [25]:
goals = int("15")

In [26]:
check("2.6", lambda: goals, 15)

PASS     2.6


### 2.7 SyntaxError: Python cannot even read the code
Nothing in the cell runs. The caret `^` points at or just after the problem. Every `if`, `for` and `def` line needs a colon.

In [28]:
pts = 102
if pts > 100:
    verdict = "big score"

In [29]:
check("2.7", lambda: verdict, "big score")

PASS     2.7


### 2.8 The dangerous one: no error, wrong answer

Errors are friendly; they tell you something is wrong. The bug below raises nothing, because it is valid Python written with R habits. Fix it so `first_quarter` is the first element and `first_two` the first two.

In [31]:
quarters = [3, 5, 4, 3]
first_quarter = quarters[0]
first_two = quarters[0:2]

In [32]:
check("2.8 first_quarter", lambda: first_quarter, 3)
check("2.8 first_two", lambda: first_two, [3, 5])

PASS     2.8 first_quarter
PASS     2.8 first_two


## 3. Handling errors on purpose

| R | Python |
| --- | --- |
| `tryCatch(expr, error = function(e) ...)` | `try:` ... `except ValueError:` ... |
| `stop("message")` | `raise ValueError("message")` |
| `warning("message")` | `import warnings` then `warnings.warn("message")` |

Always name the exception you expect. A bare `except:` hides every bug, including typos.

In [33]:
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return float("nan")


print(safe_divide(10, 4), safe_divide(10, 0))

2.5 nan


In [34]:
def parse_score(text):
    """Turn '15.12' into (15, 12). Raise a clear error for anything else."""
    parts = text.split(".")
    if len(parts) != 2:
        raise ValueError(f"Expected 'goals.behinds', got {text!r}")
    return int(parts[0]), int(parts[1])


print(parse_score("15.12"))

try:
    parse_score("15-12")
except ValueError as e:
    print("Caught:", e)

(15, 12)
Caught: Expected 'goals.behinds', got '15-12'


### Your turn 3.1

Write `safe_int(text)`. It returns `int(text)` when that works, and `None` when `int()` raises a `ValueError`.

In [37]:
def safe_int(text):
    """Turn text into an int, or return None if it can't be parsed."""
    try:
        return int(text)
    except ValueError:
        return None

In [38]:
check("3.1 number", lambda: safe_int("42"), 42)
check("3.1 word", lambda: safe_int("forty"), None)
check("3.1 decimal", lambda: safe_int("15.12"), None)

PASS     3.1 number
PASS     3.1 word
PASS     3.1 decimal


### Your turn 3.2

Write `total_from_text(text)`. It uses `parse_score()` from above to return the total score (goals × 6 + behinds), or `None` if the text cannot be parsed. Both of the bad inputs in the checks raise a `ValueError` somewhere inside `parse_score()`.

In [44]:
def total_from_text(text):
    try:
        goals, behinds = parse_score(text)
    except ValueError:
        return None
    return goals * 6 + behinds

In [45]:
check("3.2 valid", lambda: total_from_text("15.12"), 102)
check("3.2 wrong separator", lambda: total_from_text("15-12"), None)
check("3.2 not numbers", lambda: total_from_text("ab.cd"), None)

PASS     3.2 valid
PASS     3.2 wrong separator
PASS     3.2 not numbers


## 4. The debugger

For this part, leave the notebook and open `buggy_ladder.py` in VS Code. It runs without errors but prints the wrong ladder. Find the bug **with the debugger**, not by reading the code.

1. In Terminal, run `uv run python sessions/buggy_ladder.py` and compare the output with the expected ladder it prints.
2. In VS Code, click in the margin to the left of the line `if home_total > away_total:` to set a red breakpoint.
3. Press **F5** and choose **Python File** if asked. The script pauses at the breakpoint.
4. Look at the **Variables** panel on the left. Check `home_total` and `away_total` against the match you can see in `m`.
5. Press **F5** (Continue) to move to the next match. Repeat until the numbers stop making sense.
6. In the **Debug Console** at the bottom, type expressions such as `m["away_goals"] * 6 + m["away_behinds"]` to test ideas, like `browser()` in R.
7. Stop debugging (Shift+F5), fix the bug, and run the script again.

Two more tools for later:

- In a notebook, the dropdown next to a cell's Run button has **Debug Cell**, which works the same way.
- Typing `breakpoint()` on a line of code pauses there when run. This is the closest match to R's `browser()`.

## Wrap-up

1. Note which errors you hit most often. NameError and IndexError usually top the list for R users.
2. Commit: `git add sessions/` then `git commit -m "Session 4: errors and debugging"` then `git push`.